# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")

In [ ]:
docs = loader.load()
from IPython.display import display, Markdown

display(Markdown(docs[0].page_content))

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

# Connect to AI Model Client that we are going to use and initialize it
client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')

# The BaseModel class declaration - we will hold the retrieve data in objects of this class
class DocumentInfo(BaseModel):
    title: str
    author: str
    relevance: str=Field(description="A statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.")
    summary: str=Field(description="A concise and succinct summary no longer than 1000 tokens, using the tone.")
    tone: str=Field(description="The tone used to produce the summary.")
    inputTokens: list[int]=Field(description="Number of input tokens (obtain this from the response object).")
    outputTokens: list[int]=Field(description="Number of tokens in output (obtain this from the response object).")

modelName = "gpt-4o-mini"
tone = "Victorian English"
temper = 1.1
devPrompt = f"Extract the information from contents that user has passed. Use text_format format if it's a document evaluation. Use tone of {tone}"
userPrompt = f"Hi! Please evaluate this document information: {docs[0]}"

response = client.responses.parse(
    model=modelName,
    input=[
        {"role": "system", "content": devPrompt},
        {
            "role": "user",
            "content": userPrompt,
        },
    ],
    temperature=temper,
    text_format=DocumentInfo,
)

structuredOutput = response.output_parsed
print(structuredOutput)

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.models import GPTModel

modelGPT = GPTModel(
    model="gpt-4o-mini",
    temperature=temper,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

test_case = LLMTestCase(input=userPrompt, actual_output=structuredOutput.summary)
metric_SM = SummarizationMetric(
    threshold=0.5,
    model=modelGPT,
    assessment_questions=[
        "Is summary based on all content of the text?",
        "Does summary mention the author?",
        "Does summary reflect on the topic of the text?",
        "Does summary reflect the meaning that author wanted to convey?",
        "Does summary contain logical mistakes that contradict with the text?"
    ]
)

metric_Clar = GEval(
    model=modelGPT,
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids inapropriate words.",
        "Check whether ideas that are presented are easy to follow.",
        "Locate any confusing parts that reduce understanding.",
        "Assert that the output is coherent"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

metric_Ton = GEval(
    model=modelGPT,
    name="Professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Assert that the tonality is appropriate"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

metric_Safety = GEval(
    model=modelGPT,
    name="PII Leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts.",
        "Assert that the output content is safe to show to any general public user"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
)

metric_SM.measure(test_case)
metric_Clar.measure(test_case)
metric_Ton.measure(test_case)
metric_Safety.measure(test_case)

print(f'SummarizationScore: {metric_SM.score}')
print(f'SummarizationReason: {metric_SM.reason}')
print(f'CoherenceScore: {metric_Clar.score}')
print(f'CoherenceReason: {metric_Clar.reason}')
print(f'TonalityScore: {metric_Ton.score}')
print(f'TonalityReason: {metric_Ton.reason}')
print(f'SafetyScore: {metric_Safety.score}')
print(f'SafetyReason: {metric_Safety.reason}')

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:

evaluation = f"Evaluation Data: Summarization [Score: {metric_SM.score}; Reason: {metric_SM.reason}], Coherence [Score: {metric_Clar.score}; Reason: {metric_Clar.reason}], Tonality: [Score: {metric_Ton.score}; Reason: {metric_Ton.reason}], Safety: [Score: {metric_Safety.score}; Reason: {metric_Safety.reason}]"
promptEnhanced = f"Use the user's input and produce the summary given the evaluation data from previous prompt ({evaluation}), re-produce the summary and fix the mistakes. Here is the summary: {structuredOutput.summary}. Make sure to fill the summary into text_format summary field"

response = client.responses.parse(
    model=modelName,
    input=[
        {"role": "system", "content": promptEnhanced},
        {"role": "user", "content": userPrompt},
    ],
    text_format=DocumentInfo
)

print(response.output_parsed)

# Re-evaluate the new summary
old_SM = metric_SM.score
old_Clar = metric_Clar.score
old_Ton = metric_Ton.score
old_Safety = metric_Safety.score

# Evaluating the result based on old user prompt
test_case = LLMTestCase(input=userPrompt, actual_output=response.output_parsed.summary)
metric_SM.measure(test_case)
metric_Clar.measure(test_case)
metric_Ton.measure(test_case)
metric_Safety.measure(test_case)

print(f'SummarizationScore: {metric_SM.score} -- {"GOT BETTER" if metric_SM.score > old_SM else "STAYED SAME" if metric_SM.score == old_SM else "GOT WORSE"}')
print(f'SummarizationReason: {metric_SM.reason}')
print(f'CoherenceScore: {metric_Clar.score}-- {"GOT BETTER" if metric_Clar.score > old_Clar else "STAYED SAME" if metric_Clar.score == old_Clar else "GOT WORSE"}')
print(f'CoherenceReason: {metric_Clar.reason}')
print(f'TonalityScore: {metric_Ton.score}-- {"GOT BETTER" if metric_Ton.score > old_Ton else "STAYED SAME" if metric_Ton.score == old_Ton else "GOT WORSE"}')
print(f'TonalityReason: {metric_Ton.reason}')
print(f'SafetyScore: {metric_Safety.score}-- {"GOT BETTER" if metric_Safety.score > old_Safety else "STAYED SAME" if metric_Safety.score == old_Safety else "GOT WORSE"}')
print(f'SafetyReason: {metric_Safety.reason}')

Please, do not forget to add your comments.

---
#### MY COMMENTS
***I did get better output, this is wild how you can configure things using plain text with LLMs.
For this Assignment's scope these controls are enough, but maybe I would put some additional checks in case if user is trying to use a Cross-Scripting attacks on my model inside of the content he/she provide, so I can detect these too.***


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
